# Motor-Imagery EEG — full benchmark on a Colab GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ol1sa/Motor-Imagery-EEG-/blob/main/notebooks/colab_gpu_benchmark.ipynb)

Runs the **deep models** (EEGNet + Denoising-EEGNet, incl. the ablation) that are too slow on a laptop, plus the classical baselines, under both **within-subject** and **Leave-One-Subject-Out** CV.

**Before you start:** `Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU`.

**Honest caveat:** the GPU accelerates the deep models, but the first run still has to **download EEGMMIDB (~3 GB) and preprocess each subject (ICA, CPU-bound)** — budget ~30–40 s/subject. So the cohort-size knob below is the main lever on total runtime. 40 subjects + all 5 models + both CV schemes fits comfortably in one Colab session; all 109 is feasible but long (consider Colab Pro / a background run).

## 1. Confirm the GPU

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo 'No GPU — set Runtime > Change runtime type > GPU'

## 2. Clone the repo

In [ ]:
import os
REPO = 'https://github.com/ol1sa/Motor-Imagery-EEG-.git'
if not os.path.isdir('Motor-Imagery-EEG-'):
    !git clone -q $REPO
%cd Motor-Imagery-EEG-
!git pull -q  # pick up the latest if you've pushed changes

## 3. Install only what Colab lacks
Colab already ships GPU PyTorch, NumPy, SciPy, scikit-learn, pandas and matplotlib. We deliberately **do not** reinstall from `requirements.txt` (its CPU-pinned torch / NumPy<2 would replace the GPU build). We install just MNE, pyRiemann and statsmodels on top of Colab's stack.

In [ ]:
!pip install -q mne pyriemann statsmodels pooch
import torch, mne, pyriemann
print('torch', torch.__version__, '| CUDA available:', torch.cuda.is_available())
print('mne', mne.__version__, '| pyriemann', pyriemann.__version__)

## 4. Configure the run
We start from the tuned `configs/binary.yaml` and override just the cohort size and (optionally) the model list. The deep models pick up the GPU automatically via `mibci.models.torch_utils.get_device()`.

In [ ]:
import yaml

N_SUBJECTS = 40        # bump toward 109 if you have time / Colab Pro
MODELS = ['csp_lda', 'csp_svm', 'riemann_lr', 'eegnet', 'denoising_eegnet']

cfg = yaml.safe_load(open('configs/binary.yaml'))
cfg['name'] = 'colab'
cfg['data']['subjects'] = list(range(1, N_SUBJECTS + 1))
cfg['model']['names'] = MODELS
yaml.safe_dump(cfg, open('configs/colab.yaml', 'w'), sort_keys=False)
print(f'colab config: {N_SUBJECTS} subjects, models = {MODELS}')

## 5. Run the benchmark
`-u` keeps the log live so you can watch download/preprocess/training progress. Within-subject first (cheaper), then LOSO.

In [ ]:
!PYTHONPATH=src python -u -m mibci.run --config configs/colab.yaml --experiment binary --cv within

In [ ]:
!PYTHONPATH=src python -u -m mibci.run --config configs/colab.yaml --experiment binary --cv loso

### (optional) 4-class experiment
Uncomment to also run the 4-class task (left/right fist + both fists + both feet). Heavier — downloads the second run group.

In [ ]:
# fc = yaml.safe_load(open('configs/fourclass.yaml'))
# fc['name'] = 'colab'; fc['data']['subjects'] = list(range(1, N_SUBJECTS + 1)); fc['model']['names'] = MODELS
# yaml.safe_dump(fc, open('configs/colab_4c.yaml', 'w'), sort_keys=False)
# !PYTHONPATH=src python -u -m mibci.run --config configs/colab_4c.yaml --experiment fourclass --cv within

## 6. Show the results

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display

for tag in ['colab_binary_within', 'colab_binary_loso']:
    f = f'artifacts/summary_{tag}.csv'
    if glob.glob(f):
        print(f'\n===== {tag} =====')
        display(pd.read_csv(f))
        display(pd.read_csv(f'artifacts/stats_{tag}.csv'))

for png in sorted(glob.glob('artifacts/*colab*within*.png')):
    print(png); display(Image(png))

## 7. Save the results back
Either download a zip, or push to GitHub (needs a personal-access token — see the commented cell).

In [ ]:
from google.colab import files
!zip -qr mibci_colab_results.zip artifacts/*colab*
files.download('mibci_colab_results.zip')

In [ ]:
# Push results straight back to GitHub (optional).
# 1) Make a fine-grained PAT at github.com/settings/tokens with 'Contents: read & write' on this repo.
# 2) Fill TOKEN below (don't commit it!).
#
# TOKEN = ''  # paste, run, then clear this cell
# !git config user.name 'ol1sa' && git config user.email 'olisaogbue@icloud.com'
# !git add -f artifacts/*colab*
# !git commit -q -m 'Add Colab GPU benchmark results (5 models, within + LOSO)'
# !git push -q https://$TOKEN@github.com/ol1sa/Motor-Imagery-EEG-.git main
# print('pushed')